# Chạy crawler arXiv trên Colab (notebook dùng chung cho cả nhóm)

**Mỗi người dùng bản riêng:** *File → Save a copy in Drive*, rồi mở bản copy của mình.
Notebook không chứa key, mật khẩu hay thông tin của riêng ai: mọi thứ cá nhân điền ở **Bước 1** hoặc để trong *Secrets*.

**Làm một lần trước khi chạy**
1. *Runtime → Change runtime type* → **CPU** → *Save*.
2. Bấm 🔑 (*Secrets*) ở thanh bên trái, thêm (và bật *Notebook access* cho notebook này):
   - `S2_API_KEY`: key Semantic Scholar **của riêng bạn**, tự tạo tự để vào.
     Không có key vẫn chạy được nhưng phần references chậm hơn (3 s mỗi request, hay bị 429 hơn).
   - `ARXIV_PROXIES` (*tuỳ chọn*, chỉ khi bật `DUNG_PROXY`): proxy Webshare của bạn, cách nhau bằng dấu phẩy,
     dạng `http://user:pass@ip:port` hoặc `ip:port:user:pass`.
3. **Không** dán key hay mật khẩu vào ô code nào.

**⚠️ Dung lượng Drive và an toàn dữ liệu (đọc kỹ)**
- Mọi file crawler ghi ra tính vào dung lượng của **tài khoản Google bạn chọn khi kết nối Drive ở Bước 2**, kể cả khi ghi vào
  thư mục nhóm do người khác chia sẻ. Tài khoản trường thường chỉ có 20 GB; cả khoảng 5000 ID cộng các lần chạy thử
  có thể làm đầy. Xem trước ở *drive.google.com/settings/storage*, thiếu thì dùng tài khoản khác nhiều dung lượng hơn.
- Khi Drive **đầy**, Colab vẫn chạy và log trên Colab vẫn tăng nhưng file **không lên được Drive**. Thấy băng đỏ
  *Storage full* thì dừng ngay (■), dọn chỗ (xoá thư mục chạy thử thừa, **dọn thùng rác**) rồi mới chạy tiếp.
- Trong lúc crawler chạy: **không** xoá, đổi tên hay di chuyển thư mục dữ liệu (mọi file sẽ báo `FileNotFoundError`).
- Chạy xong làm **Bước 9** (đẩy hết lên Drive) rồi mới tắt runtime.

**Cách chạy:** lần lượt từ trên xuống (Shift + Enter). Mỗi thư mục dữ liệu chỉ được **một** runtime chạy tại một lúc. **Không được chạy RUN ALL**

## Bước 1. Điền thông tin (mỗi người chỉ sửa 3 dòng đầu)
- **Dữ liệu chính thức của nhóm nằm trong `02_trial/`** (không dùng `01_crawl` nữa): mỗi người một thư mục
  `02_trial/<THANH_VIEN>_<ID đầu>_<ID cuối>`, ví dụ `02_trial/AQuan_2605-12338_2605-17337`.
- `TU`, `DEN` **để trống = chạy cả khoảng giao: đây là bản chính thức.** Điền một đoạn nhỏ (ví dụ 100 ID) = chạy thử,
  ra một thư mục riêng, xoá được sau khi thử xong.
- Phần **"chung cho cả nhóm"** đã điền sẵn, không cần sửa.
- `FILE_MOI`: để `"github"` khi code mới đã push lên `BRANCH`. Chưa push thì `"tai_len"` (chọn các file `.py` mới từ máy)
  hoặc đường dẫn một thư mục Drive chứa các file `.py` mới mà người viết code chia sẻ.
- `DUNG_PROXY` / `DUNG_WARP`: đã bật sẵn. Proxy cần secret `ARXIV_PROXIES` của riêng bạn (không có thì tự đi thẳng);
  WARP do notebook tự cài trên Colab. Colab cấm kết nối proxy ngoài: nếu runtime bị ngắt bất thường thì đặt cả hai thành `False`.

In [ ]:
# ===================== MỖI NGƯỜI TỰ ĐIỀN =====================
THANH_VIEN  = ""      # tên của bạn, đứng đầu tên thư mục dữ liệu, ví dụ "AQuan", "GBao"
KHOANG_GIAO = ""      # khoảng được giao, chép 4 cột trong sheet, dạng "YYYY-MM <số đầu> YYYY-MM <số cuối>"
TU, DEN     = "", ""  # để trống = cả khoảng giao (bản chính thức); chạy thử một đoạn thì điền "YYMM.NNNNN"

# ============ CHUNG CHO CẢ NHÓM (đã điền sẵn, không cần sửa) ============
NHOM_DIR   = "/content/drive/MyDrive/group11-data"
REPO_URL   = "https://github.com/mituleefam/group11-arxiv-project.git"
BRANCH     = "quan-download-robust"
FILE_MOI   = "github"   # code mới chưa push lên BRANCH thì: "tai_len" | thư mục Drive chứa các file .py mới
DUNG_PROXY = True       # cần secret ARXIV_PROXIES (Webshare của riêng bạn); không có thì tự đi thẳng mạng Colab
DUNG_WARP  = True       # notebook tự cài WARP trên Colab ở Bước 4, không cần cài gì trên máy mình
# =========================================================================

## Bước 2. Kết nối Drive và kiểm tra thông tin
Ô 1 hiện cửa sổ: chọn đúng tài khoản có thư mục dữ liệu của nhóm (`NHOM_DIR`) → *Allow*.
**Dữ liệu tính vào dung lượng của tài khoản bạn chọn ở đây** (xem cảnh báo ở đầu notebook).
Nếu thư mục đó do người khác chia sẻ: trên Drive bấm chuột phải → *Thêm lối tắt vào Drive* (vào *My Drive*).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, re

def dung(msg):
    raise SystemExit("⚠️ " + msg)

def doc_id(s):
    """'YYMM.NNNNN' -> (YYMM, NNNNN); None nếu sai dạng."""
    k = re.fullmatch(r"\s*(\d{4})\.(\d{4,5})\s*", s or "")
    return (int(k[1]), int(k[2])) if k else None

def ma(t):
    yymm, n = t
    return f"{yymm:04d}.{n:05d}" if yymm >= 1501 else f"{yymm:04d}.{n:04d}"

THANH_VIEN = THANH_VIEN.strip()
for ten, gia_tri in (("NHOM_DIR", NHOM_DIR), ("REPO_URL", REPO_URL), ("BRANCH", BRANCH)):
    if "<" in gia_tri or not gia_tri.strip():
        dung(f"Điền {ten} ở Bước 1 (thay phần <...>): hỏi người gửi notebook nếu không biết")
if not re.fullmatch(r"[A-Za-z0-9_-]+", THANH_VIEN):
    dung('Điền THANH_VIEN ở Bước 1, ví dụ "AQuan" (chỉ chữ không dấu, số, - và _)')
k = re.fullmatch(r"\s*(\d{4})-(\d{2})\s+(\d+)\s+(\d{4})-(\d{2})\s+(\d+)\s*", KHOANG_GIAO)
if not k:
    dung('KHOANG_GIAO phải có dạng "YYYY-MM <số đầu> YYYY-MM <số cuối>" (4 cột trong sheet)')
dau, cuoi = (int(k[1][2:] + k[2]), int(k[3])), (int(k[4][2:] + k[5]), int(k[6]))
if dau > cuoi:
    dung("KHOANG_GIAO: ID đầu phải đứng trước ID cuối")
if not os.path.isdir(NHOM_DIR):
    dung(f"Không thấy {NHOM_DIR}. Thư mục do người khác chia sẻ: chuột phải -> 'Thêm lối tắt vào Drive' rồi chạy lại.")

tu = doc_id(TU) if TU.strip() else dau
den = doc_id(DEN) if DEN.strip() else cuoi
if tu is None or den is None:
    dung('TU và DEN phải có dạng "YYMM.NNNNN" (hoặc để trống cả hai)')
if not dau <= tu <= den <= cuoi:
    dung(f"Cần {ma(dau)} <= TU <= DEN <= {ma(cuoi)} (nằm trong khoảng giao)")
CA_KHOANG = (tu, den) == (dau, cuoi)
DATA = f"{NHOM_DIR}/02_trial/{THANH_VIEN}_{ma(tu).replace('.', '-')}_{ma(den).replace('.', '-')}"
PHAM_VI = f"--start {ma(tu)} --end {ma(den)}"

os.makedirs(DATA, exist_ok=True)
# Danh sách ID (chỉ lập được khi đầu và cuối cùng tháng) để validate.py kiểm tra không sót ID nào
if tu[0] == den[0]:
    SO_ID = den[1] - tu[1] + 1
    DS_ID = f"{DATA}/ds_id.txt"
    with open(DS_ID, "w", encoding="utf-8") as f:
        f.write("\n".join(ma((tu[0], n)) for n in range(tu[1], den[1] + 1)) + "\n")
else:
    SO_ID, DS_ID = None, None

print(f"Thành viên: {THANH_VIEN} | " + ("CẢ khoảng giao: bản chính thức" if CA_KHOANG else "một đoạn: chạy thử"))
print(f"Khoảng crawl: {ma(tu)} .. {ma(den)}" + (f" ({SO_ID} ID)" if SO_ID else " (qua nhiều tháng)"))
print("Ghi vào:", DATA)
print("Tham số cho main.py:", PHAM_VI)
print(f'Để theo dõi trong terminal (Bước 6), dán dòng này trước: D="{DATA}"')
print("Đã có dữ liệu từ lần trước (sẽ chạy tiếp):", os.path.isfile(f"{DATA}/work/paper_manifest.jsonl"))

## Bước 3. Lấy code
Ô 1 lấy code nhánh `BRANCH` (và trả mọi file về bản GitHub, nên chạy lại bao nhiêu lần cũng được).
Ô 2 nạp file `.py` mới theo `FILE_MOI`. Với `"tai_len"`: chọn **cùng lúc** các file `.py` mới (giữ Ctrl khi bấm);
đừng chọn `proxies.txt`. Bản `references.py` mới lấy references của nhiều bài trong một request và in tiến độ mỗi 100 bài
(chưa push thì nạp cùng; Bước 5 ô 1 cho biết đang dùng bản nào). Runtime mới thì phải nạp lại.

In [ ]:
REPO = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
os.chdir("/content")
if not os.path.isdir(f"{REPO}/.git"):
    !git clone -q -b {BRANCH} {REPO_URL} {REPO}
else:
    !git -C {REPO} checkout -q -- .
    !git -C {REPO} fetch -q origin {BRANCH}
    !git -C {REPO} checkout -q -B {BRANCH} FETCH_HEAD
os.chdir(REPO)
!git log --oneline -1
!pip install -q -r src/requirements.txt

In [ ]:
import glob, shutil

def chep(ten, du_lieu):
    if not ten.endswith(".py"):
        print("bỏ qua", ten, "(chỉ nhận file .py)")
        return
    dich = f"{REPO}/src/{ten}"
    print(("thay " if os.path.exists(dich) else "thêm ") + ten)
    with open(dich, "wb") as f:
        f.write(du_lieu)

if FILE_MOI == "github":
    print(f"Dùng nguyên code nhánh {BRANCH}")
elif FILE_MOI == "tai_len":
    from google.colab import files
    TAM = "/content/tai_len"
    shutil.rmtree(TAM, ignore_errors=True)
    os.makedirs(TAM)
    os.chdir(TAM)  # files.upload() lưu thêm một bản ở thư mục hiện tại: để nó ở đây, không lẫn vào repo
    try:
        up = files.upload()
    finally:
        os.chdir(REPO)
    if not up:
        dung("Chưa chọn file nào: chạy lại ô này")
    for ten, du_lieu in up.items():
        chep(ten, du_lieu)
elif os.path.isdir(FILE_MOI):
    ds = sorted(glob.glob(f"{FILE_MOI.rstrip('/')}/*.py"))
    if not ds:
        dung(f"Không có file .py nào trong {FILE_MOI}")
    for p in ds:
        with open(p, "rb") as f:
            chep(os.path.basename(p), f.read())
else:
    dung(f'FILE_MOI = "{FILE_MOI}" không hợp lệ: dùng "github", "tai_len" hoặc một thư mục có thật trên Drive')

## Bước 4. Proxy và WARP (tuỳ chọn)
Hai ô này chỉ làm gì khi `DUNG_PROXY` / `DUNG_WARP` bật ở Bước 1, và chỉ có tác dụng với `rate_limiter.py` bản có proxy.
- Proxy chỉ dùng cho lệnh gọi **API arXiv**; tải file không qua proxy (đỡ tốn 1 GB của Webshare);
  Semantic Scholar không bao giờ qua proxy (giới hạn của S2 tính theo key, không theo IP).
- WARP: wgcf tạo tài khoản WARP miễn phí, wireproxy mở SOCKS5 ở `127.0.0.1:40000` (cổng `rate_limiter.py` dò lúc crawler
  khởi động). Ô tự kiểm tra `warp=on`; không kết nối được thì tự tắt, crawler đi thẳng mạng Colab.

Đổi công tắc hay sửa secret thì chạy lại Bước 1, 2 (ô 2), 4 và 5.

In [ ]:
from google.colab import userdata

def chuan_hoa(p):
    p = p.strip()
    if p and "://" not in p and p.count(":") == 3:  # dạng Webshare ip:port:user:pass
        ip, port, user, pw = p.split(":")
        p = f"http://{user}:{pw}@{ip}:{port}"
    return p

os.environ.pop("ARXIV_PROXIES", None)
if os.path.isfile(f"{REPO}/src/proxies.txt"):  # rate_limiter.py tự đọc file này: không để nó lẫn vào
    os.remove(f"{REPO}/src/proxies.txt")
if not DUNG_PROXY:
    print("Proxy: tắt (API arXiv đi thẳng mạng Colab)")
else:
    try:
        raw = userdata.get("ARXIV_PROXIES")
        ds = [chuan_hoa(p) for p in raw.replace("\n", ",").replace(" ", ",").split(",") if p.strip()]
        os.environ["ARXIV_PROXIES"] = ",".join(ds)
        print(f"Proxy: đã nạp {len(ds)} proxy")
    except Exception as exc:
        print(f"Proxy: không nạp được ARXIV_PROXIES ({type(exc).__name__}): API arXiv đi thẳng mạng Colab. "
              "Kiểm tra secret và Notebook access.")

In [ ]:
# WARP: wgcf + wireproxy (WireGuard chạy trong userspace, không cần TUN). Phải chạy TRƯỚC Bước 6.
import socket, subprocess, time

WARP_DIR = "/content/warp"
WARP_PORT = 40000
WGCF_URL = "https://github.com/ViRb3/wgcf/releases/download/v2.3.1/wgcf_2.3.1_linux_amd64"
WIREPROXY_URL = "https://github.com/pufferffish/wireproxy/releases/download/v1.1.3/wireproxy_linux_amd64.tar.gz"
SOCKS = f"socks5h://127.0.0.1:{WARP_PORT}"

def cong_mo():
    try:
        with socket.create_connection(("127.0.0.1", WARP_PORT), timeout=0.5):
            return True
    except OSError:
        return False

def tat_warp():
    subprocess.run(["pkill", "-f", "chay_warp.sh"])     # vòng tự khởi động lại
    subprocess.run(["pkill", "-f", "wireproxy -s -c"])
    for _ in range(20):
        if not cong_mo():
            return
        time.sleep(0.25)

def trace():
    r = subprocess.run(["curl", "-s", "--max-time", "15", "-x", SOCKS,
                        "https://www.cloudflare.com/cdn-cgi/trace"], capture_output=True, text=True)
    return dict(line.split("=", 1) for line in r.stdout.splitlines() if "=" in line)

def chay(cmd, **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if r.returncode:
        dong = [l.strip() for l in (r.stderr or r.stdout).splitlines() if l.strip()] or ["?"]
        loi = [l for l in dong if any(k in l.lower() for k in ("error", "fail", "denied", "429", "lookup"))]
        raise RuntimeError(f"{os.path.basename(cmd[0])} {cmd[1] if len(cmd) > 1 else ''}: {(loi or dong)[0][-200:]}")

tat_warp()  # chạy lại ô này thì tắt bản cũ trước
if not DUNG_WARP:
    print("WARP: tắt (tải file đi thẳng mạng Colab)")
else:
    try:
        !pip install -q "requests[socks]"
        os.makedirs(WARP_DIR, exist_ok=True)
        wgcf, wp = f"{WARP_DIR}/wgcf", f"{WARP_DIR}/wireproxy"
        if not os.path.isfile(wgcf):
            chay(["curl", "-sSfL", "-o", wgcf, WGCF_URL])
            os.chmod(wgcf, 0o755)
        if not os.path.isfile(wp):
            chay(["curl", "-sSfL", "-o", f"{WARP_DIR}/wp.tar.gz", WIREPROXY_URL])
            chay(["tar", "-xzf", f"{WARP_DIR}/wp.tar.gz", "-C", WARP_DIR, "wireproxy"])
            os.chmod(wp, 0o755)
        if not os.path.isfile(f"{WARP_DIR}/wgcf-profile.conf"):
            chay([wgcf, "register", "--accept-tos"], cwd=WARP_DIR)
            chay([wgcf, "generate"], cwd=WARP_DIR)
        with open(f"{WARP_DIR}/wireproxy.conf", "w") as f:
            f.write(f"WGConfig = {WARP_DIR}/wgcf-profile.conf\n\n[Socks5]\nBindAddress = 127.0.0.1:{WARP_PORT}\n")
        with open(f"{WARP_DIR}/chay_warp.sh", "w") as f:  # wireproxy có tắt giữa chừng thì 2 s sau tự bật lại
            f.write(f"while true; do {wp} -s -c {WARP_DIR}/wireproxy.conf >> {WARP_DIR}/wireproxy.log 2>&1; sleep 2; done\n")
        subprocess.Popen(["bash", f"{WARP_DIR}/chay_warp.sh"], start_new_session=True,
                         stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        info = {}
        for _ in range(10):  # cổng mở chưa đủ: phải đi được ra Internet qua WARP
            time.sleep(2)
            info = trace() if cong_mo() else {}
            if info.get("warp") in ("on", "plus"):
                break
        if info.get("warp") in ("on", "plus"):
            print(f"WARP: đang chạy ở {SOCKS}, IP ra ngoài {info.get('ip')} ({info.get('colo')})")
        else:
            tat_warp()
            print("WARP: không kết nối được (Colab có thể chặn UDP 2408). Đã tắt; crawler tự bỏ qua WARP.")
    except Exception as exc:
        tat_warp()
        print(f"WARP: không bật được ({exc}). Crawler tự bỏ qua WARP, tải đi thẳng mạng Colab.")

## Bước 5. Kiểm tra và nạp key
Ô 1 kiểm tra code và đường đi mà **không gửi gì tới arXiv**. Ô 2 (chỉ khi bật proxy/WARP) thử từng đường qua
`api.ipify.org` và `speed.cloudflare.com`: proxy nào báo **HỎNG** thì xoá khỏi secret rồi chạy lại từ Bước 4;
WARP chậm hơn mạng Colab nhiều thì tắt `DUNG_WARP`. Ô 3 nạp key Semantic Scholar.

In [ ]:
import subprocess, sys
KIEM_TRA = r"""
import inspect, re, sys
sys.path.insert(0, "src")
import downloader, references, rate_limiter as rl
an = lambda p: re.sub(r"//[^@/]*@", "//***@", p) if p else "mạng Colab"
lim = (downloader.run.__kwdefaults__ or {}).get("limiter")
song_song = "workers" in inspect.signature(downloader.run).parameters
if lim is not None:
    ket_noi, nhip = getattr(lim, "connections", 1), getattr(lim, "mean_interval", lim.min_interval)
    print(f"Tải /src/: bộ giới hạn '{lim.name}', {ket_noi} kết nối, trung bình {nhip:g} s/request"
          + (", tải nhiều bài cùng lúc" if song_song else ", tải từng bài một"))
print(f"API arXiv: {rl.ARXIV.min_interval:g} s/request")
print("References:", "nhiều bài mỗi request (/paper/batch), bản mới" if hasattr(references, "plan_groups")
      else "từng bài một, bản cũ (muốn dùng bản mới: nạp references.py mới qua FILE_MOI)")
if hasattr(rl, "PROXIES"):
    print("Đường cho API arXiv:", [an(p) for p in rl.PROXIES])
    print("Đường cho tải file:", [an(p) for p in rl._UNLIMITED_LIST])
    print("Tạm bỏ proxy hỏng:", "CÓ" if hasattr(rl, "_quarantine") else "KHÔNG (nên dùng rate_limiter.py bản đã sửa)")
else:
    print("CODE_CHUA_CO_PROXY: mọi request đi thẳng mạng Colab")
"""
kq = subprocess.run([sys.executable, "-c", KIEM_TRA], capture_output=True, text=True, encoding="utf-8", cwd=REPO)
print(kq.stdout + kq.stderr[-3000:])
if kq.returncode:
    dung("Code lỗi khi nạp: xem thông báo ở trên (thiếu file mới? chạy lại Bước 3)")
if (DUNG_PROXY or DUNG_WARP) and "CODE_CHUA_CO_PROXY" in kq.stdout:
    print("⚠️ DUNG_PROXY/DUNG_WARP đang bật nhưng code này chưa có xoay proxy: nạp rate_limiter.py mới qua FILE_MOI")

In [ ]:
# Thử từng đường (không gửi gì tới arXiv): IP ra ngoài và tốc độ tải. Chỉ chạy khi bật proxy/WARP.
KIEM_DUONG = r"""
import re, sys, time
sys.path.insert(0, "src")
import requests, rate_limiter as rl

def an(p):
    return re.sub(r"//[^@/]*@", "//***@", p) if p else "mạng Colab"

if not hasattr(rl, "PROXIES"):
    sys.exit("Code này chưa có xoay proxy")
print("IP ra ngoài của từng đường cho API arXiv:")
hong = []
for p in rl.PROXIES:
    try:
        ip = requests.get("https://api.ipify.org", timeout=15, proxies={"http": p, "https": p} if p else None).text
        print(f"   OK    {an(p):40} -> {ip}")
    except Exception as e:
        if p:
            hong.append(an(p))
        print(f"   HỎNG  {an(p):40} -> {type(e).__name__}")
print("\nTốc độ tải 10 MB qua các đường dùng để tải file:")
for p in rl._UNLIMITED_LIST:
    try:
        t = time.perf_counter()
        n = len(requests.get("https://speed.cloudflare.com/__down?bytes=10000000", timeout=60,
                             proxies={"http": p, "https": p} if p else None).content)
        print(f"   {an(p):40} {n / 1e6 / (time.perf_counter() - t):6.1f} MB/s")
    except Exception as e:
        print(f"   {an(p):40} lỗi {type(e).__name__}")
if hong:
    print(f"\n{len(hong)} proxy hỏng: {', '.join(hong)}")
    print("-> xoá khỏi secret ARXIV_PROXIES rồi chạy lại từ Bước 4. Proxy hỏng không làm dừng crawler")
    print("   (nó bị bỏ qua 5 phút mỗi lần lỗi), nhưng mỗi lần lỗi tốn một request phải thử lại.")
"""
if not (DUNG_PROXY or DUNG_WARP):
    print("Bỏ qua: proxy và WARP đều tắt")
elif "CODE_CHUA_CO_PROXY" in kq.stdout:
    print("Bỏ qua: code này chưa có xoay proxy")
else:
    kq2 = subprocess.run([sys.executable, "-c", KIEM_DUONG], capture_output=True, text=True, encoding="utf-8", cwd=REPO)
    print(kq2.stdout + kq2.stderr[-2000:])

In [ ]:
# Nạp key Semantic Scholar từ Secrets (không in key ra)
try:
    os.environ["S2_API_KEY"] = userdata.get("S2_API_KEY")
    print("Đã nạp S2_API_KEY")
except Exception as exc:
    os.environ.pop("S2_API_KEY", None)
    print(f"Không nạp được S2_API_KEY ({type(exc).__name__}): vẫn chạy được, phần references chậm hơn")

## Bước 6. Chạy
Để **tab này mở** và **máy không ngủ**. Trong lúc ô này chạy, **đừng chạy ô khác** (sẽ phải xếp hàng chờ). Dòng cuối có `Crawler return code: N`:

| N | Nghĩa | Làm gì |
|---|---|---|
| 0 | Xong | Sang Bước 7 (còn vài file `failed` thì 1–2 giờ sau chạy lại để lấy nốt) |
| 3 | arXiv từ chối liên tục 30 phút | 1–2 giờ sau chạy lại (xem *Nếu bị ngắt* ở cuối) |
| 130 | Bạn bấm ■ | Chạy lại khi muốn: làm tiếp phần còn thiếu |
| 2 | Có crawler khác đang dùng thư mục này | Tắt runtime/tab kia; nếu nó đã chết thì chờ 3 phút rồi chạy lại |

Những dòng sau là **bình thường**, cứ để chạy, crawler tự xử lý:
- `[arXiv] HTTP 429/503 … pausing all arXiv requests for N s`: arXiv đang quá tải, crawler dừng rồi thử lại
  (mỗi lần lâu gấp đôi, tối đa 5 phút); thấy `… requests in a row succeeded: back to …` là đã hồi phục.
- `[download] … failed: HTTP 503`: file đó tạm gác, được thử lại ở các vòng sau và ở lần chạy sau.
- `[proxy] … bỏ qua 5 phút`: proxy đó lỗi, request đi đường khác. Cùng một proxy hiện mãi thì xoá nó khỏi secret.
- Runtime tự ngắt bất thường khi đang bật proxy/WARP: nhiều khả năng do Colab chặn, tắt 2 công tắc ở Bước 1.

**Theo dõi tiến độ**
- Bước tải: dòng `[download] x/y … about N min left`. Bước tải xong mà ô chưa kết thúc là đang chờ references
  (với bản mới thường là chờ tra ngày nộp arXiv của các bài được trích dẫn, xem dòng dưới).
- References: với `references.py` mới có dòng `[references] x/y | … | k batch calls, m papers one by one | about N min left`
  mỗi 100 bài; phần Semantic Scholar thường xong trong vài phút. Sau dòng
  `[references] Semantic Scholar done; finishing arXiv date lookups` là phần tra ngày nộp arXiv (100 ID mỗi request,
  3 s một request), có thể lâu hơn; dòng `[references] finished in …` là xong hẳn. Với bản cũ (không in gì),
  mở *Cửa sổ dòng lệnh* (terminal) ở góc dưới, dán dòng `D="…"` mà Bước 2 in ra, rồi chạy:
  ```bash
  a=$(wc -l < $D/logs/references_log.csv); sleep 60; b=$(wc -l < $D/logs/references_log.csv); echo "$b dòng, +$((b - a)) bài/phút"
  ```

**Sự cố thường gặp**
- Drive báo *Storage full*: bấm ■, dọn chỗ và dọn thùng rác, rồi chạy lại ô này (crawler làm tiếp phần còn thiếu).
- Hàng loạt `FileNotFoundError … No such file or directory`: thư mục dữ liệu bị xoá/đổi tên/di chuyển khi đang chạy.
  Khôi phục nó (xem *Thùng rác* trên Drive), mở runtime mới, chạy lại từ Bước 2.

In [ ]:
!python src/benchmark.py --data-dir "{DATA}" --interval 2 --disk-interval 300 --heartbeat-seconds 60 -- python src/main.py {PHAM_VI} --data-dir "{DATA}"

## Bước 7. Kiểm tra dữ liệu
Không có dòng `FAIL` là đạt. `PENDING` ở phiên bản `not_found` nghĩa là arXiv không có file nguồn cho phiên bản đó (404),
không phải lỗi crawler; `PENDING` ở phiên bản `failed` thì chạy lại Bước 6 sau 1–2 giờ để lấy nốt.
Ô dưới ghi các phiên bản 404 vào `not_found_can_kiem_tra.csv` trong thư mục dữ liệu. Mở vài trang `https://arxiv.org/abs/<id><version>`
kiểm tra (thường là bản đã rút, không có *TeX Source*); đúng thì đổi tên file thành `verified_404.csv` rồi chạy lại ô:
các dòng đó chuyển thành `WARN` (404 đã kiểm tra), ghi vào báo cáo như một ngoại lệ có ghi nhận.

In [ ]:
DS_ARG = f'--assigned-ids "{DS_ID}"' if DS_ID else ""
V404 = f"{DATA}/verified_404.csv"
if os.path.isfile(V404):
    DS_ARG += f' --verified-404 "{V404}"'
    print("Dùng danh sách 404 đã kiểm tra:", V404)
!python src/validate.py --data-dir "{DATA}" {DS_ARG} --finished --progress-every 500
import pandas as pd
v = pd.read_csv(f"{DATA}/logs/validation_report.csv", dtype=str)
khac = v[v.status != "PASS"]
print("Tất cả đều PASS" if khac.empty else f"{len(khac)} dòng không phải PASS:")
if not khac.empty:
    display(khac[["arxiv_id", "version", "check_name", "status", "error_message"]])
nf = khac[(khac.status == "PENDING") & khac.error_message.str.contains("not_found", na=False)][["arxiv_id", "version"]]
if not nf.empty:
    nf.to_csv(f"{DATA}/not_found_can_kiem_tra.csv", index=False)
    print(f"{len(nf)} phiên bản arXiv trả 404: đã ghi {DATA}/not_found_can_kiem_tra.csv để kiểm tra")

## Bước 8. Tốc độ và số lần arXiv từ chối

In [ ]:
import glob, json
import pandas as pd
if os.path.isfile("src/group_benchmark.py"):
    !python src/group_benchmark.py {THANH_VIEN}="{DATA}" --out "{DATA}/tong_ket"
!python src/benchmark_runs.py --logs-dir "{DATA}/logs"
L = f"{DATA}/logs"

def doc(ten):
    p = f"{L}/{ten}"
    return pd.read_csv(p) if os.path.isfile(p) and os.path.getsize(p) else pd.DataFrame()

rl_ = doc("run_log.csv")
if not rl_.empty:
    print("\n== Thời gian từng bước (run_log.csv) ==")
    display(rl_[["stage", "seconds", "items", "note"]])
dl = doc("download_log.csv")
if not dl.empty:
    cuoi = dl.drop_duplicates(["arxiv_id", "version"], keep="last")
    print("Phiên bản theo trạng thái:", cuoi.status.value_counts().to_dict())
    tai = rl_[rl_.stage == "download"] if not rl_.empty else rl_
    if not tai.empty and len(cuoi):
        print(f"Bước tải: {tai.seconds.sum():.0f} s cho {len(cuoi)} phiên bản = {tai.seconds.sum() / len(cuoi):.2f} s/phiên bản")
th = doc("arxiv_throttle_log.csv")
if th.empty:
    print("\narXiv không từ chối lần nào")
else:
    print("\narXiv từ chối/nghỉ:", th.event.value_counts().to_dict(),
          f"| tổng thời gian nghỉ {pd.to_numeric(th.total_paused_seconds, errors='coerce').max():.0f} s")
tom = sorted(glob.glob(f"{L}/performance_summary_*.json"))
if tom:
    s = json.load(open(tom[-1], encoding="utf-8"))
    wall = s.get("wall_seconds") or 0
    print(f"\nLần chạy cuối: {wall:.0f} s, mã thoát {s.get('return_code')}, "
          f"RAM đỉnh {(s.get('ram_sampled_peak_bytes') or 0) / 2**20:.0f} MB")
    if wall and SO_ID and s.get("started_with_fresh_data_dir"):
        print(f"Ước chừng cho 5000 ID (nhân tuyến tính, chỉ tham khảo): {wall / SO_ID * 5000 / 3600:.1f} giờ")

## Bước 9. Kết thúc: đẩy hết dữ liệu lên Drive
Colab ghi vào Drive qua bộ đệm. Chạy ô này **trước khi tắt runtime** để chắc mọi file đã lên Drive
(sau đó muốn chạy ô khác thì phải chạy lại Bước 2). Rồi mở Drive web xem `logs/` có thời gian sửa gần đây.

In [ ]:
from google.colab import drive
drive.flush_and_unmount()
print("Đã đẩy hết lên Drive và gỡ Drive. Có thể tắt runtime.")

### Nếu bị ngắt (Colab ngắt, mất mạng, mã 3, bấm ■)
Không mất gì: mọi thứ đã lên Drive vẫn còn. Mở lại notebook (runtime mới), **giữ nguyên Bước 1** (cùng `THANH_VIEN`, `KHOANG_GIAO`,
`TU`, `DEN` thì cùng thư mục), chạy lần lượt Bước 2 → 6 (Bước 3 phải nạp lại file mới nếu dùng `FILE_MOI`).
- Bước 2 phải in `Đã có dữ liệu từ lần trước (sẽ chạy tiếp): True`; in `False` là đang trỏ sang thư mục khác, đừng chạy kẻo tải lại từ đầu.
- Crawler tự làm tiếp phần còn thiếu (dòng `[download] … already done`); log ghi nối thêm và `benchmark_runs.py` cộng các lần chạy.
- Báo mã 2 (khoá của lần trước): chờ 3 phút rồi chạy lại Bước 6.
- Chỉ chạy **một** tab/runtime mỗi lúc cho một thư mục.